# Chapter 6: Reasoning -- Multi-Hop Inference on the Manifold

**Monograph:** *How to Build Policy-Governed AI Agents*

---

## Learning Objectives

By the end of this notebook you will be able to:

1. Use `GeometricReasoner` to perform multi-hop inference across a GMS knowledge store.
2. Find scored reasoning paths connecting concepts (e.g., data profile to model recommendation).
3. Validate multi-step reasoning chains via per-link plausibility and holonomy defect.
4. Detect contradictions among model findings using pairwise tension energy.
5. Explain why a claim is or is not plausible using paths and tension analysis.

## Prerequisites

- Chapters 2 and 5 completed (GMS store fundamentals and agent memory).
- The domain knowledge document `governed_agent/knowledge/data_science_practices.md`.

---
## Preamble

---
## Key Concepts

| Term | Definition |
|---|---|
| **GeometricReasoner** | Agent-friendly wrapper that composes GMS primitives (scoring, transport, search) into multi-hop reasoning, chain validation, contradiction detection and explanation. Contains no tensor code of its own. |
| **Plausibility threshold** ($\tau$) | Geodesic distance cutoff (default 1.5): a triple with $d < \tau$ is classified as plausible. Requires calibration for each store's training quality. |
| **Multi-hop path** | A sequence of relation rotors $R_{r_1}, \ldots, R_{r_L}$ composed on the manifold $\mathcal{S}^{m-1}$. The path score measures how close the composed transport lands to the target entity. |
| **Holonomy defect** ($H$) | $\|R_{\text{path}} R_{\text{direct}}^{-1} - I\|_F$: measures rotational mismatch between the multi-hop path and a direct relation. $H \approx 0$ indicates consistent reasoning. |
| **Tension energy** | Pairwise distance in *u*-space (logical embedding) between entities. $E \approx 0$ = agreement, $E \approx \sqrt{2}$ = irrelevance, $E \approx 2$ = contradiction. |
| **Contradiction detection** | Flags entity pairs whose tension energy exceeds threshold $\tau_c$. Threshold choice controls the trade-off between precision and recall. |
| **explain\_why verdict** | Multi-faceted judgment combining direct scoring, path search and tension analysis. Verdicts: *supported*, *unsupported* or *contradicted*. |
| **Triple-set confidence** | Aggregate score $\exp(-\bar{d})$ over a collection of triples. Useful for batch assessment of reasoning chains or findings sets. |

---

In [ ]:
# === Preamble (all notebooks) ===
import os
import warnings
warnings.filterwarnings("ignore")

import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}, CUDA: {torch.cuda.is_available()}")


---
## Lab 6.1 --- Multi-Hop Reasoning for Model Selection

In Chapter 2 we ingested the data science practices document and explored the six GMS primitives individually. Now we compose them into **reasoning**: multi-hop path finding and chain validation that let an agent navigate from a data condition to a model recommendation through intermediate concepts.

### Step 1: Build the Knowledge Store

We reuse the data science knowledge store from Chapter 2. If it was already saved we load it; otherwise we train from scratch.

In [ ]:
from knowlytix.knowledge.config import DocGMSConfig
from knowlytix.knowledge.store import GMSExpertStore
from knowlytix.knowledge.ingest import ingest_document

STORE_PATH = "governance_monograph/lab_stores/ds_knowledge"
MD_PATH = "governed_agent/knowledge/data_science_practices.md"

config = DocGMSConfig(store_path=STORE_PATH)
config.train.epochs = 500

import shutil
if os.path.exists(STORE_PATH):
    shutil.rmtree(STORE_PATH)

store = GMSExpertStore(config, device=DEVICE)
result = ingest_document(store, MD_PATH, llm=None, config=config, device=DEVICE)

stats = store.stats()
print("\n=== Knowledge Store ===")
print(f"  Entities:  {stats['entities']}")
print(f"  Relations: {stats['relations']}")
print(f"  Triples:   {stats['triples']}")
print(f"  ENM:       {stats['enm_entries']}")

### Step 2: Create the GeometricReasoner

The `GeometricReasoner` wraps the store's model, transport layer and search machinery into an agent-friendly API. It does not duplicate any geometry -- it delegates to `GeometricSearch`, `RelationalTransport` and the store primitives.

In [3]:
# The GeometricReasoner now REQUIRES a CalibratedThresholds object
# on construction -- every cut (plausibility, tau_ent, tau_contra)
# must be derived from data, not hardcoded. So we defer building
# the reasoner until after calibration in Step 2b.

print("Store vocabulary")
print(f"  Entities:  {len(store.adapter.entity_to_idx)}")
print(f"  Relations: {len(store.adapter.relation_to_idx)}")
print(f"  Relation names: {sorted(store.adapter.relation_to_idx.keys())}")


Store vocabulary
  Entities:  244
  Relations: 38
  Relation names: ['fails', 'has_affected_models', 'has_applicability', 'has_attribute', 'has_condition', 'has_description', 'has_gate_type', 'has_interpretable', 'has_interpretation', 'has_maximum_rmse_ratio', 'has_minimum_auc', 'has_minimum_f1', 'has_minimum_precision', 'has_minimum_r-squared', 'has_minimum_recall', 'has_missing_pattern', 'has_must_test', 'has_order', 'has_phase', 'has_precedes', 'has_priority', 'has_range', 'has_rationale', 'has_recommended_method', 'has_recommended_model', 'has_regularized', 'has_regulation', 'has_release_decision', 'has_required_action', 'has_required_checks', 'has_requires', 'has_risk_tiers', 'has_threshold', 'has_when_to_use', 'in_section', 'is_strong', 'is_weak', 'passes']


### Step 2b: Calibrate Thresholds and Build the Reasoner

The GMS library ships a calibrated judge --- `knowlytix.harness.testing.GMSJudge`. Its `calibrate()` method fits a **logistic regression** for each measure (geodesic, path transport, tension, holonomy, phase) against tail-corrupted negatives, then derives the decision threshold from the coefficient and intercept. This is the same LR-based diagnostic banking practitioners know from credit scoring and MRM.

For contradiction detection we need two cuts on tension energy: `tau_ent` (agreement / irrelevance) and `tau_contra` (irrelevance / contradiction). The judge returns them under the strict 3-class protocol: `tau_ent < tau_contra` or the calibration is flagged as *degenerate* and the reasoner refuses to operate geometrically (it can still fire on schema-declared oppositions). **No hardcoded default thresholds are used** --- if calibration does not produce usable cuts, detection raises rather than silently substituting a default.

We wrap the judge output in a `CalibratedThresholds` dataclass and pass it to the reasoner.


In [ ]:
from knowlytix.harness.testing import GMSJudge
from knowlytix.harness.governance.reasoner import (
    CalibratedThresholds, GeometricReasoner,
)

judge = GMSJudge(store)
print("Calibrating measures (logistic regression, tail-corrupted negatives)...")
judge.calibrate()

thresholds = CalibratedThresholds.from_judge(judge)

print("\n=== Calibrated Thresholds ===")
print(f"  tau_plausibility (geodesic)  : {thresholds.tau_plausibility}")
print(f"  tau_ent          (tension 0/1): {thresholds.tau_ent}")
print(f"  tau_contra       (tension 1/2): {thresholds.tau_contra}")
print(f"  tension_status               : {thresholds.tension_status}")
print(f"  tension_ok                   : {thresholds.tension_ok}")

# Construct the reasoner. Passing a raw float threshold raises.
reasoner = GeometricReasoner(store, thresholds, device=DEVICE)
print("\nGeometricReasoner constructed with data-driven thresholds.")


### Step 3: Plausibility Checking

The simplest reasoning primitive: "Is this triple plausible?" The reasoner scores the triple on $S^{m-1}$ and compares against a threshold. True facts from the document should score low (plausible); fabricated facts should score high (implausible).

In [5]:
# Test plausibility on known-true triples from the store
all_triples = list(store.doc_graph.triples)
print(f"Total triples in store: {len(all_triples)}\n")

# Score a sample of true triples
print("=== Known-True Triples (from the document) ===")
true_checked = 0
for h, r, t in all_triples:
    result = reasoner.is_plausible(h, r, t)
    if result is not None:
        plausible, score = result
        tag = "PLAUSIBLE" if plausible else "IMPLAUSIBLE"
        print(f"  ({h}, {r}, {t})")
        print(f"    geodesic = {score:.4f}  [{tag}]")
        true_checked += 1
    if true_checked >= 8:
        break

# Now test fabricated triples
import random
random.seed(42)
entity_names = list(store.adapter.entity_to_idx.keys())

print("\n=== Fabricated Triples (wrong tail) ===")
fab_checked = 0
for h, r, t in all_triples[:20]:
    wrong_t = random.choice([e for e in entity_names if e != t])
    result = reasoner.is_plausible(h, r, wrong_t)
    if result is not None:
        plausible, score = result
        tag = "PLAUSIBLE" if plausible else "IMPLAUSIBLE"
        print(f"  ({h}, {r}, {wrong_t})")
        print(f"    geodesic = {score:.4f}  [{tag}]")
        fab_checked += 1
    if fab_checked >= 6:
        break

Total triples in store: 334

=== Known-True Triples (from the document) ===
  (High-dimensional, many categoricals, has_recommended_model, GradientBoosting)
    geodesic = 1.0340  [PLAUSIBLE]
  (High-dimensional, many categoricals, has_rationale, Handles categorical features natively, built-in regularization)
    geodesic = 1.0848  [PLAUSIBLE]
  (High-dimensional, many categoricals, in_section, model_selection_guidelines)
    geodesic = 1.0130  [PLAUSIBLE]
  (High-dimensional, many categoricals, has_recommended_model, XGBoost)
    geodesic = 1.1170  [PLAUSIBLE]
  (High-dimensional, many categoricals, has_recommended_model, RandomForest)
    geodesic = 0.9960  [PLAUSIBLE]
  (High-dimensional, many categoricals, has_rationale, Ensemble averaging reduces variance, handles mixed types)
    geodesic = 1.1537  [PLAUSIBLE]
  (Small dataset below 1000 rows, has_recommended_model, LogisticRegression)
    geodesic = 1.0836  [PLAUSIBLE]
  (Small dataset below 1000 rows, has_rationale, Low varianc

### Step 4: Multi-Hop Path Finding

The agent needs to answer questions like "How do I get from a high-dimensional dataset to a specific model recommendation?" This requires traversing multiple edges in the knowledge graph. The reasoner's `find_path` method runs BFS over the adjacency graph and scores each discovered path using geodesic distance on $S^{m-1}$.

In [6]:
# Find paths from data conditions to model recommendations
# First, let us see what entities are related to data profiles and models
print("=== Entities containing 'high' or 'dimensional' ===")
data_entities = [e for e in entity_names
                 if "high" in e.lower() or "dimensional" in e.lower()
                 or "categor" in e.lower()]
print(f"  {data_entities[:15]}")

print("\n=== Entities that are models ===")
model_entities = [e for e in entity_names
                  if any(m in e for m in ["GradientBoosting", "RandomForest",
                         "LogisticRegression", "XGBoost", "SVM",
                         "DecisionTree", "LinearRegression", "Ridge",
                         "Lasso", "ElasticNet"])]
print(f"  {model_entities[:15]}")

# Pick source and target groups
source = set(data_entities[:5]) if data_entities else set(entity_names[:3])
target = set(model_entities[:5]) if model_entities else set(entity_names[-3:])
print(f"\nSource entities: {source}")
print(f"Target entities: {target}")

=== Entities containing 'high' or 'dimensional' ===
  ['Add missing indicator category', 'Categorical', 'Categorical features present', 'Handles categorical features natively, built-in regularization', 'High', 'High cardinality features', 'High discriminative power on tabular data', 'High-dimensional sparse', 'High-dimensional, many categoricals', 'High-risk', 'High-risk (credit, lending)', 'High-risk and medium-risk', 'Yes for high-risk']

=== Entities that are models ===
  ['DecisionTree', 'DecisionTree, RandomForest, GradientBoosting', 'DecisionTree/Yes (depth<=3)/No', 'ElasticNet', 'ElasticNet/Yes/Yes (L1+L2)', 'GradientBoosting', 'GradientBoosting/No/Yes (shrinkage)', 'Lasso', 'Lasso/Yes/Yes (L1)', 'LinearRegression', 'LinearRegression, LogisticRegression', 'LinearRegression/Yes/No', 'LogisticRegression', 'LogisticRegression, SVM, KNN', 'LogisticRegression, SVM, LinearRegression']

Source entities: {'Categorical', 'Add missing indicator category', 'Categorical features present', '

In [7]:
# Run multi-hop path search
result = reasoner.find_path(source, target, max_hops=4)

if result is not None and result.paths:
    print("=== Multi-Hop Path Search Results ===")
    print(f"  Paths found: {len(result.paths)}")
    print(f"  Max hops:    {result.n_hops}")
    print(f"  Best geodesic: {result.best_geodesic:.4f}" if result.best_geodesic else "")
    print()

    for i, path in enumerate(result.paths[:8]):
        chain_str = " -> ".join(path.entity_names)
        rels_str = " | ".join(path.relation_names) if path.relation_names else "N/A"
        print(f"  Path {i+1}: {chain_str}")
        print(f"    Relations: {rels_str}")
        print(f"    Geodesic:  {path.geodesic:.4f}  |  Hops: {path.hops}")
        print()
else:
    print("No paths found. Trying with broader entity sets...")
    # Fallback: use first few entities from the triple set
    heads = set(h for h, r, t in all_triples[:20])
    tails = set(t for h, r, t in all_triples[-20:])
    result = reasoner.find_path(heads, tails, max_hops=5)
    if result and result.paths:
        print(f"  Found {len(result.paths)} paths with broader search")
        for i, path in enumerate(result.paths[:5]):
            print(f"  Path {i+1}: {' -> '.join(path.entity_names)}")
            print(f"    Geodesic: {path.geodesic:.4f}  |  Hops: {path.hops}")
    else:
        print("  No paths found even with broader search.")

=== Multi-Hop Path Search Results ===
  Paths found: 5
  Max hops:    3
  Best geodesic: 1.3319

  Path 1: Categorical features present -> DecisionTree, RandomForest, GradientBoosting
    Relations: has_affected_models
    Geodesic:  1.3319  |  Hops: 1

  Path 2: Handles categorical features natively, built-in regularization -> High-dimensional, many categoricals -> model_selection_guidelines -> DecisionTree/Yes (depth<=3)/No
    Relations: has_rationale | in_section | is_weak
    Geodesic:  1.5291  |  Hops: 3

  Path 3: Handles categorical features natively, built-in regularization -> High-dimensional, many categoricals -> model_selection_guidelines -> ElasticNet
    Relations: has_rationale | in_section | in_section
    Geodesic:  1.5466  |  Hops: 3

  Path 4: Handles categorical features natively, built-in regularization -> High-dimensional, many categoricals -> model_selection_guidelines -> DecisionTree
    Relations: has_rationale | in_section | in_section
    Geodesic:  1.6617  |

### Step 5: Validate a Reasoning Chain

An agent that proposes a multi-step plan needs to verify that the chain of reasoning is internally consistent. The `validate_chain` method scores each link individually and computes the holonomy defect for the overall chain.

In [8]:
# Build a reasoning chain from actual triples in the store
# Pick a chain of connected triples
print("=== Building a Reasoning Chain from the Knowledge Graph ===\n")

# Build adjacency for chain construction
from collections import defaultdict
adj = defaultdict(list)
for h, r, t in all_triples:
    adj[h].append((r, t))

# Find a chain of length 3+ by BFS from a popular head entity
chain_triples = []
visited = set()
current = all_triples[0][0]
visited.add(current)

for _ in range(4):
    neighbors = adj.get(current, [])
    found_next = False
    for rel, tail in neighbors:
        if tail not in visited:
            chain_triples.append((current, rel, tail))
            visited.add(tail)
            current = tail
            found_next = True
            break
    if not found_next:
        break

print(f"Chain of {len(chain_triples)} hops:")
for h, r, t in chain_triples:
    print(f"  ({h}) --[{r}]--> ({t})")

# Validate the chain
validation = reasoner.validate_chain(chain_triples)

print("\n=== Chain Validation Results ===")
print(f"  All plausible: {validation.all_plausible}")
print(f"  Mean score:    {validation.mean_score:.4f}" if validation.mean_score else "  Mean score:    N/A")

for link in validation.links:
    tag = "OK" if link.plausible else "FAIL"
    score_str = f"{link.score:.4f}" if link.score is not None else "N/A"
    print(f"  ({link.head}) --[{link.relation}]--> ({link.tail})")
    print(f"    score = {score_str}  [{tag}]")

if validation.holonomy_defect is not None:
    print(f"\n  Holonomy defect: {validation.holonomy_defect:.4f}")
    print(f"  Consistent:      {validation.consistent}")

=== Building a Reasoning Chain from the Knowledge Graph ===

Chain of 2 hops:
  (High-dimensional, many categoricals) --[has_recommended_model]--> (GradientBoosting)
  (GradientBoosting) --[has_interpretable]--> (No)



=== Chain Validation Results ===
  All plausible: True
  Mean score:    1.1052
  (High-dimensional, many categoricals) --[has_recommended_model]--> (GradientBoosting)
    score = 1.0340  [OK]
  (GradientBoosting) --[has_interpretable]--> (No)
    score = 1.1763  [OK]

  Holonomy defect: 1.5273
  Consistent:      False


### Step 6: Recommend Next Step

The `recommend_next` method answers "given where I am in the workflow, what should I do next?" It wraps `link_predict` to rank all entities by geodesic distance after applying a relation rotor.

In [9]:
# What should come after data loading? Use the most likely workflow relation.
all_relations = sorted(store.adapter.relation_to_idx.keys())
print(f"Available relations: {all_relations}\n")

# Try different starting entities and relations
test_queries = []

# Find entities that appear as heads in the triples
head_counts = defaultdict(int)
for h, r, t in all_triples:
    head_counts[h] += 1
popular_heads = sorted(head_counts, key=head_counts.get, reverse=True)[:5]

for head in popular_heads:
    for rel in all_relations[:3]:
        preds = reasoner.recommend_next(head, rel, top_k=5)
        if preds:
            test_queries.append((head, rel, preds))

print("=== Recommend Next: Top Predictions ===\n")
for head, rel, preds in test_queries[:6]:
    print(f"  Query: ({head}, {rel}, ?)")
    for entity, dist in preds[:5]:
        print(f"    -> {entity:40s}  geodesic = {dist:.4f}")
    print()

Available relations: ['fails', 'has_affected_models', 'has_applicability', 'has_attribute', 'has_condition', 'has_description', 'has_gate_type', 'has_interpretable', 'has_interpretation', 'has_maximum_rmse_ratio', 'has_minimum_auc', 'has_minimum_f1', 'has_minimum_precision', 'has_minimum_r-squared', 'has_minimum_recall', 'has_missing_pattern', 'has_must_test', 'has_order', 'has_phase', 'has_precedes', 'has_priority', 'has_range', 'has_rationale', 'has_recommended_method', 'has_recommended_model', 'has_regularized', 'has_regulation', 'has_release_decision', 'has_required_action', 'has_required_checks', 'has_requires', 'has_risk_tiers', 'has_threshold', 'has_when_to_use', 'in_section', 'is_strong', 'is_weak', 'passes']



=== Recommend Next: Top Predictions ===

  Query: (Demographic, fails, ?)
    -> model_selection_guidelines_DecisionTree/Yes (depth<=3)/No  geodesic = 1.3782
    -> model_selection_guidelines_GradientBoosting/No/Yes (shrinkage)  geodesic = 1.4286
    -> model_selection_guidelines_RandomForest/No/Yes (ensemble)  geodesic = 1.5445
    -> model_selection_guidelines_XGBoost/No/Yes (L1/L2/shrinkage)  geodesic = 1.6163

  Query: (Demographic, has_affected_models, ?)
    -> All classification models                 geodesic = 1.4679
    -> All models                                geodesic = 1.4685
    -> LogisticRegression, SVM, KNN              geodesic = 1.4979
    -> LinearRegression, LogisticRegression      geodesic = 1.5630
    -> LogisticRegression, SVM, LinearRegression  geodesic = 1.5673

  Query: (Demographic, has_applicability, ?)
    -> Credit and lending models                 geodesic = 1.5888
    -> All protected attributes                  geodesic = 1.6198

  Query: (Need int

---
## Lab 6.2 --- Contradiction Detection Across Model Findings

In a real governance scenario an agent trains multiple models and stores findings. Different models may produce conflicting claims (e.g., two models both claiming "best accuracy" or one model claiming a feature is important while another ranks it low). The reasoner's `detect_contradictions` and `explain_why` methods expose these conflicts geometrically.

### Step 1: Train Multiple Models and Store Findings

We train three sklearn classifiers on UCI Adult Income, collect their metrics and store the results as triples in a fresh GMS store.

In [10]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline

# Adult Income: 5,000-row stratified-on-sex subsample for lab speed.
X_full, y_full = fetch_openml(
    "adult", version=2, as_frame=True, return_X_y=True,
)
y_bin = (y_full == ">50K").astype(int)

sample = (
    X_full.assign(_y=y_bin)
    .groupby("sex", observed=True, group_keys=False)
    .apply(lambda g: g.sample(min(len(g), 2500), random_state=42))
    .reset_index(drop=True)
)
X = sample.drop(columns=["_y"])
y = sample["_y"]
sex = X["sex"].astype(str).values

X_train, X_test, y_train, y_test, sex_train, sex_test = train_test_split(
    X, y, sex, test_size=0.30, random_state=42, stratify=y,
)

num_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = X.select_dtypes(include=["category"]).columns.tolist()
pre = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])

models = {
    "LogisticRegression": LogisticRegression(max_iter=2000, random_state=42),
    "RandomForest":       RandomForestClassifier(n_estimators=50, max_depth=6,
                                                 random_state=42, n_jobs=-1),
    "GradientBoosting":   GradientBoostingClassifier(n_estimators=100, max_depth=3,
                                                     random_state=42),
}

results = {}
for name, clf in models.items():
    pipe = Pipeline([("pre", pre), ("clf", clf)])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]
    acc = accuracy_score(y_test, y_pred)
    f1  = f1_score(y_test, y_pred, average="binary")
    auc = roc_auc_score(y_test, y_proba)
    male = (sex_test == "Male"); female = (sex_test == "Female")  # noqa: E702
    pr_m = float(y_pred[male].mean()) if male.any() else 0.0
    pr_f = float(y_pred[female].mean()) if female.any() else 0.0
    air = (pr_f / pr_m) if pr_m > 0 else 0.0
    results[name] = {
        "accuracy": round(acc, 4),
        "f1":       round(f1, 4),
        "auc":      round(auc, 4),
        "air_sex":  round(air, 4),
    }
    print(f"{name:<22s}  acc={acc:.4f}  f1={f1:.4f}  auc={auc:.4f}  air_sex={air:.4f}")

best_model = max(results, key=lambda m: results[m]["auc"])
print(f"\nBest model by AUC: {best_model}")


LogisticRegression      acc=0.8740  f1=0.6678  auc=0.9083  air_sex=0.2837


RandomForest            acc=0.8687  f1=0.5988  auc=0.9113  air_sex=0.1840


GradientBoosting        acc=0.8887  f1=0.6980  auc=0.9192  air_sex=0.3011

Best model by AUC: GradientBoosting


### Step 2: Build a Findings Store

We now encode the model findings as triples and ingest them into a GMS store. This creates a geometric representation of the experimental results where we can detect contradictions.

In [ ]:
import tempfile
import os

# Build the Model Performance Summary table the parser handles natively
# (numeric columns -> ENM + has_<col> triples). The opposition /
# functional declarations and the verdict triples go in via direct
# doc_graph.add_triple calls AFTER parsing -- that keeps the ENM
# machinery of the ingestion pipeline while giving us precise control
# over the triples that drive mining and calibration.
#
# Deliberate contradictions:
#   * GradientBoosting / XGBoost pass best_auc_gate; the others fail.
#   * LogisticRegression / GradientBoosting approve low_variance_claim;
#     RandomForest / DecisionTree reject it.
# Declared schema:
#   * passes opposite_of fails
#   * approves opposite_of rejects
#   * has_verdict is_functional true   (single verdict per model)

metrics_md = """# Model Comparison Findings -- Adult Income Classification

## Model Performance Summary

| Model | Test Accuracy | F1 | AUC | AIR (sex) |
|---|---|---|---|---|
| LogisticRegression | 0.8740 | 0.6678 | 0.9083 | 0.2837 |
| RandomForest | 0.8687 | 0.5988 | 0.9113 | 0.1840 |
| GradientBoosting | 0.8887 | 0.6980 | 0.9192 | 0.3011 |
| DecisionTree | 0.8421 | 0.5712 | 0.8843 | 0.2104 |
| XGBoost | 0.8912 | 0.7044 | 0.9205 | 0.3150 |
"""

metrics_path = tempfile.mktemp(suffix=".md")
with open(metrics_path, "w") as f:
    f.write(metrics_md)

# Parse the metrics to a DocumentGraph via the library parser.
from knowlytix.benchmark.ingest import ingest_markdown
doc_graph = ingest_markdown(metrics_path)
os.unlink(metrics_path)

# Augment the graph with verdict triples and schema declarations
# BEFORE training so mine_tension_pairs sees them at ingest time.
verdicts = [
    ("GradientBoosting",   "passes", "best_auc_gate"),
    ("XGBoost",            "passes", "best_auc_gate"),
    ("RandomForest",       "fails",  "best_auc_gate"),
    ("LogisticRegression", "fails",  "best_auc_gate"),
    ("DecisionTree",       "fails",  "best_auc_gate"),
    ("GradientBoosting",   "passes", "fairness_gate"),
    ("XGBoost",            "passes", "fairness_gate"),
    ("LogisticRegression", "passes", "fairness_gate"),
    ("RandomForest",       "fails",  "fairness_gate"),
    ("DecisionTree",       "fails",  "fairness_gate"),
    ("LogisticRegression", "approves", "low_variance_claim"),
    ("GradientBoosting",   "approves", "low_variance_claim"),
    ("RandomForest",       "rejects",  "low_variance_claim"),
    ("DecisionTree",       "rejects",  "low_variance_claim"),
]
for (h, r, t) in verdicts:
    doc_graph.add_triple(h, r, t)

# Schema declarations: live as ordinary graph triples so they are
# auditable and domain-owned -- no hardcoded vocabulary.
declarations = [
    ("passes",   "opposite_of", "fails"),
    ("approves", "opposite_of", "rejects"),
]
for (h, r, t) in declarations:
    doc_graph.add_triple(h, r, t)

# Now run the rest of the ingestion pipeline manually: build the
# GMS adapter, train, populate ENM, and wire the store state.
from knowlytix.core.train_finstructbench import GraphToGMS, train_gms, populate_enm
from knowlytix.core.graph.transport import RelationalTransport
from knowlytix.core.memory.compression import CompressionMemory
from knowlytix.core.memory.router import MemoryRouter
import shutil

findings_config = DocGMSConfig(
    store_path="governance_monograph/lab_stores/ch06_findings",
)
findings_config.train.epochs = 500
if os.path.exists(findings_config.store_path):
    shutil.rmtree(findings_config.store_path)
findings_store = GMSExpertStore(findings_config, device=DEVICE)

adapter = GraphToGMS(doc_graph)
model = train_gms(
    adapter, DEVICE,
    epochs=findings_config.train.epochs,
    batch_size=findings_config.train.batch_size,
    num_neg=findings_config.train.neg_samples,
    lr=findings_config.train.lr,
    lr_riemannian=findings_config.train.lr_riemannian,
    geometry=findings_config.geometry,
    loss=findings_config.loss,
)
model.eval()
enm = populate_enm(doc_graph, adapter)
transport = RelationalTransport(model)
compression = CompressionMemory(
    k=findings_config.memory.k, d=findings_config.geometry.m,
)
router = MemoryRouter(enm, compression)

findings_store.model = model
findings_store.adapter = adapter
findings_store.enm = enm
findings_store.doc_graph = doc_graph
findings_store.transport = transport
findings_store.compression = compression
findings_store.router = router
findings_store.markdown = metrics_md

fs = findings_store.stats()
print("\n=== Findings Store ===")
print(f"  Entities:  {fs['entities']}")
print(f"  Relations: {fs['relations']}")
print(f"  Triples:   {fs['triples']}")
print(f"  ENM:       {fs['enm_entries']}")

print("\nRelation vocabulary:")
for r in sorted(findings_store.adapter.relation_to_idx.keys()):
    print(f"  {r}")


### Step 3: Detect Contradictions

Now we use the GeometricReasoner on the findings store to detect which model entities have high tension energy -- indicating contradictory claims.

In [ ]:
# Calibrate the findings store. The corpus declared opposite_of
# triples, so label_tension produces contradict samples. The strict
# 3-class calibrator returns tau_ent < tau_contra or flags the fit
# as degenerate -- no silent collapse.
from knowlytix.harness.testing import GMSJudge

findings_judge = GMSJudge(findings_store)
print("Calibrating findings store...")
findings_judge.calibrate()

findings_thresholds = CalibratedThresholds.from_judge(findings_judge)

print("\n=== Findings store thresholds ===")
print(f"  tau_plausibility : {findings_thresholds.tau_plausibility}")
print(f"  tau_ent          : {findings_thresholds.tau_ent}")
print(f"  tau_contra       : {findings_thresholds.tau_contra}")
print(f"  tension_status   : {findings_thresholds.tension_status}")

# Build the reasoner. Construction raises if thresholds are not a
# CalibratedThresholds object; hardcoded floats are refused.
findings_reasoner = GeometricReasoner(
    findings_store, findings_thresholds, device=DEVICE,
)
print("\nFindings reasoner constructed.")

# List all entities in the findings store (for detect_contradictions)
findings_entities = list(findings_store.adapter.entity_to_idx.keys())
print(f"\nEntities in store ({len(findings_entities)}):")
for e in sorted(findings_entities):
    print(f"  {e}")


In [13]:
import math

# Compute ALL pairwise tension energies to see the full landscape
print("=== Pairwise Tension Energy Landscape ===\n")
all_tensions = findings_reasoner.get_all_tensions(findings_entities)

# Display as a table
if all_tensions:
    print(f"{'Entity A':30s} {'Entity B':30s} {'Tension':>8s}  Interpretation")
    print("-" * 85)
    for a, b, t, interp in sorted(all_tensions, key=lambda x: -x[2]):
        print(f"{a:30s} {b:30s} {t:8.4f}  {interp}")
    print(f"\n  Reference: sqrt(2) = {math.sqrt(2):.4f} (irrelevance boundary)")
else:
    print("  No pairwise tensions computed (entities may not have embeddings).")

=== Pairwise Tension Energy Landscape ===

Entity A                       Entity B                        Tension  Interpretation
-------------------------------------------------------------------------------------
GradientBoosting               RandomForest                     1.7067  contradiction
DecisionTree                   GradientBoosting                 1.7057  contradiction
RandomForest                   XGBoost                          1.7049  contradiction
DecisionTree                   XGBoost                          1.7038  contradiction
GradientBoosting               LogisticRegression               1.6240  contradiction
LogisticRegression             XGBoost                          1.6191  contradiction
LogisticRegression             RandomForest                     1.5925  contradiction
DecisionTree                   LogisticRegression               1.5888  contradiction
approves                       model_performance_summary        1.5540  contradiction
0.7044    

In [14]:
# detect_contradictions now returns two provenance streams:
#   * schema   -- deterministic from declared opposite_of triples
#   * geometric -- learned, tension > calibrated tau_contra
# Each Contradiction.provenances is a frozenset that may contain
# one or both; ``schema+geometric`` is the strongest evidence --
# the corpus declared it AND training learned it.

print("=== Detected Contradictions ===\n")
contradictions = findings_reasoner.detect_contradictions(findings_entities)

if not contradictions:
    if not findings_thresholds.tension_ok:
        print(f"  No contradictions (calibration status: "
              f"{findings_thresholds.tension_status}, geometric arm "
              f"silent).")
    else:
        print("  No contradictions detected.")
else:
    print(f"{'Pair':50s} {'Tension':>8s}  Provenance")
    print("-" * 85)
    for c in contradictions:
        pair = f"{c.entity_a} <-> {c.entity_b}"
        print(f"{pair:50s} {c.tension:8.4f}  {c.provenance}")
    print()
    n_schema = sum(1 for c in contradictions if "schema" in c.provenances)
    n_geom   = sum(1 for c in contradictions if "geometric" in c.provenances)
    n_both   = sum(
        1 for c in contradictions
        if {"schema", "geometric"} <= c.provenances
    )
    print(f"  Schema-declared  : {n_schema}")
    print(f"  Geometric  > tau : {n_geom}  (tau_contra = "
          f"{findings_thresholds.tau_contra})")
    print(f"  Both arms (audit): {n_both}")

    # Expand the first schema hit so we can see the declaration trail.
    for c in contradictions:
        if "schema" in c.provenances:
            print("\n  Example audit trail:")
            print(f"    {c.entity_a} <-> {c.entity_b}")
            print(f"    opposing relations: {c.schema_relations}")
            print(f"    shared tail: {c.shared_tail}")
            print(f"    learned tension: {c.tension:.4f}")
            print(f"    threshold (geometric arm): {c.threshold}")
            break


=== Detected Contradictions ===

Pair                                                Tension  Provenance
-------------------------------------------------------------------------------------
GradientBoosting <-> RandomForest                    1.7067  schema+geometric
GradientBoosting <-> DecisionTree                    1.7057  schema+geometric
XGBoost <-> RandomForest                             1.7049  schema+geometric
XGBoost <-> DecisionTree                             1.7038  schema+geometric
GradientBoosting <-> LogisticRegression              1.6240  schema+geometric
XGBoost <-> LogisticRegression                       1.6191  schema+geometric
LogisticRegression <-> RandomForest                  1.5925  schema+geometric
LogisticRegression <-> DecisionTree                  1.5888  schema+geometric
approves <-> model_performance_summary               1.5540  geometric
0.7044 <-> passes                                    1.5538  geometric
0.184 <-> passes                           

### Step 4: Explain Why

The `explain_why` method combines direct scoring, path search and tension analysis to build a multi-faceted explanation for a claim. Let us explain why one model is recommended over another.

In [15]:
# Use the DS knowledge store reasoner to explain model recommendations
# Pick triples from the knowledge store and explain them
print("=== explain_why: True Claims from Knowledge Store ===\n")

# Find some interesting triples to explain
explain_triples = []
for h, r, t in all_triples:
    if store.entity_exists(h) and store.entity_exists(t) and store.relation_exists(r):
        explain_triples.append((h, r, t))
    if len(explain_triples) >= 3:
        break

for h, r, t in explain_triples:
    expl = reasoner.explain_why(h, r, t, max_hops=3)
    print(f"  Claim: ({expl.head}, {expl.relation}, {expl.tail})")
    print(f"  Direct score: {expl.direct_score:.4f}" if expl.direct_score else "  Direct score: N/A")
    print(f"  Verdict: {expl.verdict}")
    if expl.supporting_paths:
        print(f"  Supporting paths: {len(expl.supporting_paths)}")
        for p in expl.supporting_paths[:2]:
            print(f"    {' -> '.join(p.entity_names)}  (geo={p.geodesic:.4f})")
    if expl.contradicting_entities:
        for c in expl.contradicting_entities:
            print(f"  Tension({c.entity_a}, {c.entity_b}) = {c.tension:.4f} [{c.interpretation}]")
    print()

=== explain_why: True Claims from Knowledge Store ===

  Claim: (High-dimensional, many categoricals, has_recommended_model, GradientBoosting)
  Direct score: 1.0340
  Verdict: supported
  Supporting paths: 1
    High-dimensional, many categoricals -> GradientBoosting  (geo=1.0340)

  Claim: (High-dimensional, many categoricals, has_rationale, Handles categorical features natively, built-in regularization)
  Direct score: 1.0848
  Verdict: supported
  Supporting paths: 1
    High-dimensional, many categoricals -> Handles categorical features natively, built-in regularization  (geo=1.0848)

  Claim: (High-dimensional, many categoricals, in_section, model_selection_guidelines)
  Direct score: 1.0130
  Verdict: supported
  Supporting paths: 1
    High-dimensional, many categoricals -> model_selection_guidelines  (geo=1.0130)



In [16]:
# Explain a fabricated (likely false) claim
print("=== explain_why: Fabricated Claim ===\n")
random.seed(99)
# Pick a true triple and swap the tail to make it false
h_true, r_true, t_true = explain_triples[0]
wrong_tail = random.choice([e for e in entity_names if e != t_true])

expl_false = reasoner.explain_why(h_true, r_true, wrong_tail, max_hops=3)
print(f"  Claim: ({expl_false.head}, {expl_false.relation}, {expl_false.tail})")
print(f"  Direct score: {expl_false.direct_score:.4f}" if expl_false.direct_score else "  Direct score: N/A")
print(f"  Verdict: {expl_false.verdict}")
if expl_false.supporting_paths:
    print(f"  Supporting paths: {len(expl_false.supporting_paths)}")
if expl_false.contradicting_entities:
    for c in expl_false.contradicting_entities:
        print(f"  Tension({c.entity_a}, {c.entity_b}) = {c.tension:.4f} [{c.interpretation}]")

=== explain_why: Fabricated Claim ===

  Claim: (High-dimensional, many categoricals, has_recommended_model, High cardinality features)
  Direct score: 1.4066
  Verdict: unsupported
  Supporting paths: 1


### Step 5: Triple-Set Confidence

The `score_triple_set` method computes an aggregate confidence score for a collection of triples: $\text{conf} = \exp(-\bar{d})$ where $\bar{d}$ is the mean geodesic distance. This is useful for scoring an entire reasoning chain or a batch of findings.

In [17]:
# Score a set of true triples
true_set = [(h, r, t) for h, r, t in all_triples
            if store.entity_exists(h) and store.entity_exists(t)
            and store.relation_exists(r)][:15]

conf_true = reasoner.score_triple_set(true_set)
print("=== Triple-Set Confidence ===")
print(f"  True triples ({len(true_set)}): conf = {conf_true:.4f}")

# Score a set of fabricated triples
random.seed(42)
fab_set = []
for h, r, t in true_set:
    wrong = random.choice([e for e in entity_names if e != t])
    fab_set.append((h, r, wrong))

conf_fab = reasoner.score_triple_set(fab_set)
print(f"  Fabricated triples ({len(fab_set)}): conf = {conf_fab:.4f}")
print(f"\n  Separation: {conf_true - conf_fab:+.4f}")
print(f"  True triples have {'higher' if conf_true > conf_fab else 'lower'} confidence.")

=== Triple-Set Confidence ===
  True triples (15): conf = 0.3199


  Fabricated triples (15): conf = 0.2198

  Separation: +0.1000
  True triples have higher confidence.


---
## Summary

In this chapter we built `GeometricReasoner`, an agent-friendly wrapper over the GMS search and transport machinery. The key capabilities demonstrated:

| Method | Purpose | Geometric Mechanism |
|---|---|---|
| `is_plausible` | Judge a single claim | Geodesic distance $d(R_r \tilde{v}_h, \tilde{v}_t) \leq \tau$ |
| `find_path` | Multi-hop path search | BFS + path scoring on $S^{m-1}$ |
| `validate_chain` | Check reasoning consistency | Per-link scoring + holonomy defect |
| `detect_contradictions` | Flag conflicting entities | Pairwise tension energy $E > \tau_c$ |
| `recommend_next` | Suggest next step | `link_predict` beam search on manifold |
| `explain_why` | Multi-faceted explanation | Combines scoring, paths and tension |
| `score_triple_set` | Aggregate confidence | $\exp(-\bar{d})$ over triple set |

The reasoner does not duplicate geometry -- it delegates to `GeometricSearch`, `RelationalTransport` and the store primitives. An agent calls these methods to make geometrically grounded decisions rather than relying on LLM intuition alone.

---
## Exercises

1. **Compare Models.** Implement a `compare_models(model_a, model_b)` method that uses ENM lookups for exact metrics and `find_path` for reasoning paths connecting the two models through shared properties. Return a structured comparison dict.

2. **Hypothesis Tester.** Build a function that takes a hypothesis (e.g., "GradientBoosting is better than LogisticRegression for imbalanced data") and uses `find_path` to find supporting evidence and `detect_contradictions` to find counter-evidence. Produce a support score and counter-evidence score.

3. **Backward Reasoning.** Given an outcome like "low accuracy," use `recommend_next` with different relations to find likely causes (e.g., insufficient preprocessing, wrong model choice, data quality issues). Chain the backward reasoning through multiple hops.